
<p style="font-size:50px">
<span style="font-size:200px">
&#9703;
</span>
Le mécanisme d'import de module
</p>

---



Sous-sections :
[Le problème de la résolution des noms](#liaison)&nbsp;|
[Le mécanisme d'import de module](#import)&nbsp;|
[Les paquetages](#paquetages)&nbsp;|
[Ligne de commande et environnement](#cmd)&nbsp;


Ce notebook a pour but de mieux comprendre le mécanisme d'import de module de Python.

<a id="liaison"></a>
### Le problème de la résolution des noms

En programmant en Python, on manipule des noms, qui sont liés à tout moment à des objets.

Par exemple dans cette instruction le nom `a` n'est pour l'instant lié à aucun objet :

In [1]:
print(a)

NameError: name 'a' is not defined

On dit que le nom `a` ne fait pas partie des noms directement accessibles.

Les noms directement accessibles sont ceux auxquels je peux accéder directement (sans avoir besoin de les préfixer par un autre nom avec la notation pointée comme dans `b.a`) et sans lever d'erreur.

On peut utiliser la fonction `vars()` sans argument pour afficher les noms directement accessibles (portée courante) :

In [2]:
list(vars())

['__name__',
 '__doc__',
 '__package__',
 '__loader__',
 '__spec__',
 '__builtin__',
 '__builtins__',
 '_ih',
 '_oh',
 '_dh',
 'In',
 'Out',
 'get_ipython',
 'exit',
 'quit',
 'open',
 '_',
 '__',
 '___',
 '__vsc_ipynb_file__',
 '_i',
 '_ii',
 '_iii',
 '_i1',
 '_i2']

Si je lie la variable `a` à un objet comme ici :

In [3]:
a = 8

alors il devient accessible :

In [4]:
print(a)

8


car il a été rajoutée dans la portée courante :

In [5]:
list(vars())

['__name__',
 '__doc__',
 '__package__',
 '__loader__',
 '__spec__',
 '__builtin__',
 '__builtins__',
 '_ih',
 '_oh',
 '_dh',
 'In',
 'Out',
 'get_ipython',
 'exit',
 'quit',
 'open',
 '_',
 '__',
 '___',
 '__vsc_ipynb_file__',
 '_i',
 '_ii',
 '_iii',
 '_i1',
 '_i2',
 '_2',
 '_i3',
 'a',
 '_i4',
 '_i5']

Le problème de la __résolution des noms__ consiste pour Python à :
- déterminer à tout moment quels sont les noms directement (ou indirectement) accessibles, et quel objet est référencé par un nom donné ;
- gérer les conflits de nommage, car le même nom peut être utilisé à différents endroits du programme (ou par différents développeurs).

Par exemple, si je veux appeler la fonction `euclidean_sim` que j'ai placée dans le fichier `./cbp/similarity/measures.py`, je dois pouvoir le dire à Python, sinon il lève une erreur :

In [6]:
a = (3,4)
b = (-3,5)
euclidean_sim(a,b)

NameError: name 'euclidean_sim' is not defined

<a id="import"></a>
### Le mécanisme d'import de module

L'import d'un module consiste à :  
- Charger le module (i.e., trouver où il est défini)
- Ajouter certaines noms du module dans la portée courante

##### Chargement d'un module

Le chargement d'un module se fait en plusieurs étapes :  
1. chercher dans le dictionnaire `sys.modules`, qui référence les modules déjà chargés ;
2. s'il est absent, chercher le module dans un des répertoires de la liste `sys.path` ;
3. lorsqu'il est trouvé, ajouter une référence dans `sys.modules`.

Par exemple, la fonction `search` n'est pas directement accessible par défaut :

In [7]:
search(r'A[TC]','AGATC')

NameError: name 'search' is not defined

parce qu'elle est définie dans le module `re`, qui n'est pas répertoriée dans la table `sys.modules` :

In [8]:
import sys
sys.modules

{'sys': <module 'sys' (built-in)>,
 'builtins': <module 'builtins' (built-in)>,
 '_frozen_importlib': <module '_frozen_importlib' (frozen)>,
 '_imp': <module '_imp' (built-in)>,
 '_thread': <module '_thread' (built-in)>,
 '_warnings': <module '_warnings' (built-in)>,
 '_weakref': <module '_weakref' (built-in)>,
 '_io': <module '_io' (built-in)>,
 'marshal': <module 'marshal' (built-in)>,
 'posix': <module 'posix' (built-in)>,
 '_frozen_importlib_external': <module '_frozen_importlib_external' (frozen)>,
 'time': <module 'time' (built-in)>,
 'zipimport': <module 'zipimport' (frozen)>,
 '_codecs': <module '_codecs' (built-in)>,
 'codecs': <module 'codecs' from '/e/opt/miniconda3/envs/l3sv/lib/python3.10/codecs.py'>,
 'encodings.aliases': <module 'encodings.aliases' from '/e/opt/miniconda3/envs/l3sv/lib/python3.10/encodings/aliases.py'>,
 'encodings': <module 'encodings' from '/e/opt/miniconda3/envs/l3sv/lib/python3.10/encodings/__init__.py'>,
 'encodings.utf_8': <module 'encodings.utf_8'

mais on peut importer le module `re` :

In [9]:
import re

Dans ce cas, un fichier `re.py` est trouvé dans un répertoire d'installation de Python et une entrée est rajoutée dans la table `sys.modules` :

In [10]:
sys.modules['re']

<module 're' from '/e/opt/miniconda3/envs/l3sv/lib/python3.10/re.py'>

Le fichier `re.py` en question est un module qui contient entre autre la définition de la fonction `search` :

In [11]:
%cat {sys.modules['re'].__file__}

#
# Secret Labs' Regular Expression Engine
#
# re-compatible interface for the sre matching engine
#
# Copyright (c) 1998-2001 by Secret Labs AB.  All rights reserved.
#
# This version of the SRE library can be redistributed under CNRI's
# Python 1.6 license.  For any other use, please contact Secret Labs
# AB (info@pythonware.com).
#
# Portions of this engine have been developed in cooperation with
# CNRI.  Hewlett-Packard provided funding for 1.6 integration and
# other compatibility work.
#

r"""Support for regular expressions (RE).

This module provides regular expression matching operations similar to
those found in Perl.  It supports both 8-bit and Unicode strings; both
the pattern and the strings being processed can contain null bytes and
characters outside the US ASCII range.

Regular expressions can contain both special and ordinary characters.
Most ordinary characters, like "A", "a", or "0", are the simplest
regular expressions; they simply match themselves.  You can
concaten

##### Ajout des noms dans la portée courante

Une fois que Python a trouvé le module, certains noms sont ajoutés dans la portée courante


In [12]:
[nom for nom in list(vars()) if not nom.startswith('_')]

['In', 'Out', 'get_ipython', 'exit', 'quit', 'open', 'a', 'b', 'sys', 're']

Le nom `re` y figure maintenant, donc on peut accéder indirectement à la fonction `search` à l'aide de la notation pointée :

In [13]:
re.search(r'A[TC]','AGATC')

<re.Match object; span=(2, 4), match='AT'>

Si l'on veut pouvoir y accéder directement (sans la notation pointée), on peut utiliser la syntaxe suivante : 

In [14]:
from re import search

Le nom `search` est alors placé dans la portée courante :

In [15]:
[nom for nom in list(vars()) if not nom.startswith('_')]

['In',
 'Out',
 'get_ipython',
 'exit',
 'quit',
 'open',
 'a',
 'b',
 'sys',
 're',
 'search']

et on peut l'utiliser directement :

In [16]:
search(r'A[TC]','AGATC')

<re.Match object; span=(2, 4), match='AT'>

On peut aller plus loin et lui donner un alias (par exemple pour la distinguer d'une autre fonction `search` qu'on aurait défini par ailleurs) :

In [17]:
from re import search as chercher

In [18]:
chercher(r'A[TC]','AGATC')

<re.Match object; span=(2, 4), match='AT'>

On voudrait calculer la valeur de $sin(\pi/6)$. 

La fonction $sin$ est implémentée dans le module `math`.

Le nom de ce module est-il directement accessible ? 
Le module `math` est-il dans la table `sys.modules` ?

Importer le module `math` et vérifier :
- qu'une entrée a été placée dans la table `sys.modules` ;
- que le nom `math` est maintenant directement accessible.

Utiliser le module `math` pour calculer $sin(\pi/6)$.


In [ ]:
# <- completer ici

<a id="paquetages"></a>
### Les paquetages

Un paquetage est un répertoire contenant un ou plusieurs modules.

Pour importer un module qui se trouve dans un paquetage, on utiliser la syntaxe suivante :

In [1]:
import paquetage.module

La constante mystère
bonjour


Où est définie la constante mystère ?

Afficher successivement :
- la constante mystère
- la variable `z` du module que nous venons d'importer

In [ ]:
# <- completer ici

Importer le module `measures` du paquetage `cbp.similarity`.

Quelles entrées ont été ajoutées à la table `sys.modules` ?

Comment rendre la fonction `euclidean_sim()` du module `measures` directement accessible ?

Utiliser cette fonction pour calculer la similarité entre les vecteurs `a` et `b`.

In [ ]:
# <- completer ici

<a id="cmd"></a>
### Ligne de commande et environnement

La variable `__name__` contient le nom du module dans lequel on se trouve.

Le module par défaut s'appelle `__main__` :

In [20]:
__name__

'__main__'

En ligne de commande, l'option -c permet d'exécuter le code entre guillemets, par exemple :

In [ ]:
!python -c 'print(48)' 

Lors de l'exécution du code d'un module ou d'un paquetage, la variable `__name__` contient le nom du module ou du paquetage : 

In [ ]:
__name__, paquetage.__name__, paquetage.module.__name__, 

En ligne de commande, l'option `-m` exécute le contenu d'un module en tant que module `__main__` :

In [ ]:
!python -m paquetage.module